# 05 - Skenario-B: K klien non-IID (Paper 4, Fase 2) - uji H3

**Tujuan:** uji **skalabilitas & derajat non-IID**. Skenario-A (nb03) hanya 2 klien
(1 per dataset). Di sini tiap dataset dipecah jadi beberapa **shard non-IID**
sehingga total K klien (mis. K=4, 6, 10). Non-IID dibuat via **partisi berbasis
label** (tiap shard condong ke proporsi attack/benign berbeda), meniru realita
klien dgn distribusi serangan tak seragam.

**Hipotesis H3:** non-IID menurunkan MCC global dibanding IID; makin banyak klien
/ makin ekstrem skew -> penurunan makin besar. Ukur SEBERAPA jauh, dan apakah
ruang SFM tetap membuat federasi bermakna (vs runtuh).

**Konsistensi:** FedAvg via `../aws/fed_node.py` (logika identik EC2). Ruang fitur
= SFM 9-fitur (dari nb01). Output: `fed_out/noniid_scaling.csv`.

> Jalankan setelah nb01. JANGAN mengarang angka.

## 0. Setup + impor fed_node

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef, f1_score, balanced_accuracy_score
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; SEED=42
rng=np.random.RandomState(SEED)
print('fed_node loaded | LAYERS', fed.LAYERS)

## 1. Muat ruang SFM (dari nb01)

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
if ok:
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    print('train CIC',Xc_tr.shape,'UNSW',Xu_tr.shape,'| GLOBAL-test',Xg_te.shape)
else:
    print('(partisi SFM belum ada - jalankan nb01)')

## 2. Pembuat partisi non-IID berbasis label (Dirichlet)

Partisi gaya FL standar: proporsi kelas tiap klien ditarik dari **Dirichlet(alpha)**.
`alpha` kecil (mis. 0.1) = sangat non-IID (tiap klien condong satu kelas); `alpha`
besar (mis. 100) ~ IID. Kita bagi gabungan train (CIC+UNSW di ruang SFM yang sama)
menjadi K klien; label skew DIATUR oleh alpha.

In [ ]:
def dirichlet_partition(X, y, K, alpha, seed=SEED):
    r=np.random.RandomState(seed); idx_by_c={c:np.where(y==c)[0] for c in np.unique(y)}
    for c in idx_by_c: r.shuffle(idx_by_c[c])
    client_idx=[[] for _ in range(K)]
    for c,idc in idx_by_c.items():
        prop=r.dirichlet([alpha]*K)
        cuts=(np.cumsum(prop)*len(idc)).astype(int)[:-1]
        for k,part in enumerate(np.split(idc, cuts)):
            client_idx[k].extend(part.tolist())
    parts={}
    for k in range(K):
        ii=np.array(client_idx[k],int); r.shuffle(ii)
        if len(ii)==0: continue
        parts[f'c{k}']=(X[ii], y[ii])
    return parts
def run_fedavg(parts, rounds=ROUNDS, epochs=LOCAL_EPOCHS, lr=LR, seed=SEED):
    zc={cid: fed.zscore_fit(X) for cid,(X,y) in parts.items()}
    Xz={cid: fed.zscore_apply(parts[cid][0], *zc[cid]) for cid in parts}
    sizes={cid: len(parts[cid][1]) for cid in parts}
    Wg=fed.init_weights(seed)
    for t in range(rounds):
        cw=[]; cs=[]
        for cid in parts:
            Wl=fed.train_local(Wg, Xz[cid], parts[cid][1], epochs=epochs, lr=lr,
                               mu=0.0, W_global=Wg, seed=seed)
            cw.append(Wl); cs.append(sizes[cid])
        Wg=fed.fedavg(cw, cs)
    return Wg
def metrics(y,yp):
    return dict(MCC=round(matthews_corrcoef(y,yp),4),
                F1=round(f1_score(y,yp,zero_division=0),4),
                bal_acc=round(balanced_accuracy_score(y,yp),4))
print('partisi non-IID + FedAvg siap')

## 3. Sweep K x alpha (ukur H3)

Grid: K in {2,4,6,10}, alpha in {0.1 (ekstrem non-IID), 1.0 (sedang), 100 (~IID)}.
Evaluasi di GLOBAL-test. z-score global train hanya utk evaluasi monitoring.

In [ ]:
res=[]
if ok:
    Xtr=np.vstack([Xc_tr,Xu_tr]); ytr=np.concatenate([yc_tr,yu_tr])
    mu_g,sd_g=fed.zscore_fit(Xtr)
    for alpha in (0.1, 1.0, 100.0):
        for K in (2,4,6,10):
            parts=dirichlet_partition(Xtr, ytr, K, alpha, seed=SEED)
            if len(parts)<2: 
                res.append({'K':K,'alpha':alpha,'n_clients':len(parts),'MCC':None,'note':'partisi degenerate'}); continue
            Wg=run_fedavg(parts)
            out,_=fed.forward(Wg, fed.zscore_apply(Xg_te,mu_g,sd_g)); yp=(out>=0.5).astype(int)
            skew=round(float(np.mean([abs(p[1].mean()-ytr.mean()) for p in parts.values()])),4)
            res.append({'K':K,'alpha':alpha,'n_clients':len(parts),'label_skew':skew, **metrics(yg_te,yp)})
            print(f'alpha={alpha} K={K}: clients={len(parts)} MCC={res[-1].get("MCC")}')
    print('sweep selesai')
else:
    print('(lewati: partisi SFM belum ada)')
print('=== SEL 3 (sweep K x alpha) SELESAI ===')

## 4. Simpan + ringkas (tabel F)

In [ ]:
if ok and len(res):
    dfres=pd.DataFrame(res); dfres.to_csv(os.path.join(OUTDIR,'noniid_scaling.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    base=dfres[(dfres['K']==2)&(dfres['alpha']==100.0)]['MCC']
    if len(base) and base.iloc[0] is not None:
        print('MCC acuan (K=2, ~IID):', base.iloc[0])
    print('H3: pada alpha kecil (non-IID ekstrem) MCC harus turun; makin besar K -> turun lebih dalam.')
else:
    print('(tak ada hasil - jalankan nb01 + sel di atas di SageMaker)')
print('=== SEL 4 (ringkas) SELESAI ===')

## 5. Catatan untuk naskah

- Hasil -> **Tabel F** (`documentation_federated.md`): MCC global vs (K, alpha).
- H3 diukur dua arah: derajat non-IID (alpha turun) & jumlah klien (K naik).
- `label_skew` = rata-rata |pos-rate klien - pos-rate global|; proksi kuantitatif
  derajat non-IID per konfigurasi (dilaporkan apa adanya).
- Logika FL identik dgn nb03/EC2; hanya partisi klien yang berbeda -> perbandingan adil.
- Jika non-IID ekstrem meruntuhkan performa, itu temuan jujur -> motivasi FedProx (nb06).